# 07 · Specialists vs one generalist (RQ4)

**Specialize.** Four LoRA experts on one frozen HMR2.0 backbone (adapters in the last 4 ViT blocks + the
SMPL head), each trained on one kind of problem, with **the same images, steps, LR and rank**:

| Expert | Rows | Data (from notebook 04's folders, no new rendering) |
|---|---|---|
| E0 Clean | 1, 16 | `B_clean` |
| E1 Occlusion | 7, 8, 10, 11 | `D_occ3d` (3D-rendered occluders) |
| E2 Quality | 20–22 | `B_clean` + blur / motion blur / low light / low resolution |
| E3 Framing | 13–15 | `B_clean` + cut-off / far / close |

**Generalist baseline** (the key comparison): one adapter with rank 4×r, so exactly the same trainable
parameters as the four experts together, trained on the union of their data for the same total steps.

**Cross-evaluation matrix:** every model on every test condition (clean, 3D occlusion, 2D paste,
quality, framing, occlusion+quality). Routing here uses the *true* condition labels (oracle); notebook 08
replaces it with the learned detector. Resumable; ~1 h on a T4.

In [ ]:
EXPERTS = ["E0", "E1", "E2", "E3"]
R_LORA, LORA_BLOCKS = 16, 4
EPOCHS, BATCH, LR, SEED = 4, 16, 1e-4, 0          # LoRA uses a higher LR than full fine-tuning (1e-5)
RUN = f"experts_r{R_LORA}_b{LORA_BLOCKS}_s{SEED}"

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# HMR2.0 without PHALP/detectron2 (the install that worked in notebook 05 on Python 3.13)
%cd /content
![ -d 4D-Humans ] || git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /content/4D-Humans
!pip install -q numpy yacs smplx pytorch_lightning timm einops omegaconf trimesh webdataset pyrender
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy
!pip install -q --no-deps .
!sed -i 's/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg)/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg, weights_only=False)/' hmr2/models/__init__.py
%cd /content
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, "/content/4D-Humans")

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models
download_models(CACHE_DIR_4DHUMANS)                           # ~2.5 GB, cached for the session

# SMPL: plain-numpy files only, chumpy never imported (the fix for the broken environment)
from pose_lib.smpl import pkl_to_npz, setup_hmr2_smpl
SMPL_NPZ = f"{PROJECT_DIR}/data/smpl_neutral.npz"
if not os.path.exists(SMPL_NPZ):
    print(pkl_to_npz(f"{PROJECT_DIR}/body_models/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", SMPL_NPZ))
setup_hmr2_smpl(SMPL_NPZ)

import torch
from pose_lib import hmr2_ft as ft, lora, metrics, stats
fk = ft.SMPLJoints(SMPL_NPZ)
print("HMR2.0 + SMPL ready")

In [ ]:
# synthetic AMASS dataset from notebook 04 (B_clean / C_paste2d / D_occ3d, train + test)
DATA = "/content/dataset"
if not os.path.exists(f"{DATA}/test/D_occ3d/labels.npz"):
    !tar -xf "{PROJECT_DIR}/data/synth_amass.tar" -C /content
# restore self-occlusion labels if notebook 06 added them in an earlier session
for c in ["B_clean", "C_paste2d", "D_occ3d"]:
    saved = f"{PROJECT_DIR}/data/test_labels_selfocc_{c}.npz"
    if os.path.exists(saved):
        shutil.copy(saved, f"{DATA}/test/{c}/labels.npz")
HAS_SELFOCC = "selfvis_joint" in np.load(f"{DATA}/test/D_occ3d/labels.npz").files
print(os.listdir(f"{DATA}/train"), "| self-occlusion labels:", HAS_SELFOCC)

In [ ]:
def fresh_model():
    m = ft.load_model()
    gc.collect(); torch.cuda.empty_cache()
    return m

In [ ]:
from pose_lib import experts as ex
MODELS = f"{PROJECT_DIR}/models/{RUN}"; RESULTS = f"{PROJECT_DIR}/results/{RUN}"
os.makedirs(MODELS, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

def model_with_adapters():
    m = fresh_model()
    lora.inject(m, lora.hmr2_targets(n_blocks=LORA_BLOCKS), EXPERTS, r=R_LORA)
    lora.add_adapter(m, "GEN", r=lora.generalist_rank(R_LORA, len(EXPERTS)))
    print("params per expert:", lora.adapter_numel(m, "E0"), "| generalist:", lora.adapter_numel(m, "GEN"),
          "| 4 experts:", sum(lora.adapter_numel(m, e) for e in EXPERTS))
    return m

model = model_with_adapters()

## Train the four experts

In [ ]:
for e in EXPERTS:
    path = f"{MODELS}/{e}.pt"
    if os.path.exists(path):
        print(f"[skip] {e}"); continue
    tr, va = ex.expert_datasets(DATA, e, seed=SEED)
    print(f"\n===== {e}: {len(tr)} train / {len(va)} val =====")
    hist = ft.train(model, None, None, fk, path, epochs=EPOCHS, batch_size=BATCH, lr=LR, seed=SEED,
                    train_ds=tr, val_ds=va, trainable=lora.trainable_fn(e),
                    save_fn=lambda m, p, e=e: lora.save_adapter(m, e, p))
    json.dump(hist, open(f"{MODELS}/{e}_history.json", "w"), indent=1)

## Train the generalist (same parameters, same total steps, union of the data)

In [ ]:
path = f"{MODELS}/GEN.pt"
if not os.path.exists(path):
    parts = [ex.expert_datasets(DATA, e, seed=SEED) for e in EXPERTS]
    tr, va = ex.ConcatSame([p[0] for p in parts]), ex.ConcatSame([p[1] for p in parts])
    hist = ft.train(model, None, None, fk, path, epochs=EPOCHS, batch_size=BATCH, lr=LR, seed=SEED,
                    train_ds=tr, val_ds=va, trainable=lora.trainable_fn("GEN"),
                    save_fn=lambda m, p: lora.save_adapter(m, "GEN", p))
    json.dump(hist, open(f"{MODELS}/GEN_history.json", "w"), indent=1)
for e in EXPERTS + ["GEN"]:
    lora.load_adapter(model, e, f"{MODELS}/{e}.pt")
lora.set_active(model, None)

## Cross-evaluation matrix: every model on every condition

In [ ]:
tests = ex.test_sets(DATA)
print({k: len(v) for k, v in tests.items()})

def run_eval(name, ds, **kw):
    f = f"{RESULTS}/{name}__{tname}.npz"
    if os.path.exists(f):
        return dict(np.load(f))
    print(f"{name:>16} on {tname}:", end=" ")
    r = ft.evaluate(model, ds, fk, **kw); np.savez(f, **r); return r

dgs_w, dgs_j = ex.DGS(model, None, mode="weighted"), ex.DGS(model, None, mode="per_joint")
R = {}
for tname, ds in tests.items():
    lora.set_active(model, None);  R[("pretrained", tname)] = run_eval("pretrained", ds)
    for e in EXPERTS + ["GEN"]:
        lora.set_active(model, e); R[(e, tname)] = run_eval(e, ds)
    lora.set_active(model, None)
    R[("DGS_oracle_weighted", tname)] = run_eval("DGS_oracle_weighted", ds, predict_fn_batch=dgs_w.oracle)
    R[("DGS_oracle_perjoint", tname)] = run_eval("DGS_oracle_perjoint", ds, predict_fn_batch=dgs_j.oracle)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
MODELS_ROWS = ["pretrained"] + EXPERTS + ["GEN", "DGS_oracle_weighted", "DGS_oracle_perjoint"]
mat = pd.DataFrame({t: {m: R[(m, t)]["pa_mpjpe"].mean() for m in MODELS_ROWS} for t in tests}).round(1)
mat.to_csv(f"{RESULTS}/cross_eval_pa_mpjpe.csv")
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(mat.values, cmap="viridis_r"); plt.colorbar(im, label="PA-MPJPE (mm)")
ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=30, ha="right")
ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.values[i, j]; best = v == mat.values[:, j].min()
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", color="w" if v > mat.values.mean() else "k",
                fontweight="bold" if best else None)
ax.set_title("Cross-evaluation (bold = best per condition)"); plt.tight_layout()
plt.savefig(f"{RESULTS}/cross_eval.png", dpi=150); plt.show(); mat

In [ ]:
# RQ4: detector + specialists vs the generalist, per condition (paired bootstrap, Holm-corrected)
rows = []
for cand in ["DGS_oracle_weighted", "DGS_oracle_perjoint"]:
    for t in tests:
        m, lo, hi = stats.paired_bootstrap(R[(cand, t)]["pa_mpjpe"], R[("GEN", t)]["pa_mpjpe"])
        p = stats.bootstrap_p(R[(cand, t)]["pa_mpjpe"], R[("GEN", t)]["pa_mpjpe"])
        rows.append({"model": cand, "test": t, "diff_vs_GEN_mm": round(m, 2), "ci": f"[{lo:+.1f}, {hi:+.1f}]", "p": p})
cmp = pd.DataFrame(rows); cmp["p_holm"] = stats.holm(cmp["p"].values).round(4)
cmp["verdict"] = np.where((cmp.p_holm < 0.05) & (cmp.diff_vs_GEN_mm < 0), "specialists better",
                 np.where((cmp.p_holm < 0.05), "generalist better", "no significant difference"))
cmp.to_csv(f"{RESULTS}/rq4_vs_generalist.csv", index=False); cmp

**Reading RQ4.** Specialists help if (a) each expert wins its home condition and (b) routed specialists
beat the generalist without losing on clean images. Oracle routing is the upper bound; notebook 08 shows how
much of it the learned detector keeps. Run with `SEED = 1, 2` to get seed spread for the final report.